This notebook is to prob (currently with the setting: CGNN-3D, rmsd_cutoff_2, random-k-fold) (linear_probes) for the affinity value and the docking score, and set one as the skyline that would specify the limitation that the embeddings and the linear prob are forcing. <br>

The pipeline to load the $X$ is the same. To make the $y$, one should read the idents of X, load the raw data and sort docking scores/affinities according to the idents from $X$'s idents.

In [9]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from kinodata.data import KinodataDockedAgnostic, KinodataDocked

from prob.paths_and_io import get_project_root, get_exp_dirs, load_X_from_pt, load_out_tensor, save_out_tensor, load_y_by_ids
from prob.prob_config import get_ds_load_config
from prob.prob_models import LINEAR_PROBES
from prob.prob_run import run_cv_search



In [3]:
prob_config = get_ds_load_config()

output_dir = prob_config['output_dir']
target_dir = prob_config['target_dir']
print(f"output_dir: {output_dir}")
print(f"target_dir: {target_dir}")

output_dir: /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold
target_dir: /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/targets


In [4]:
IDS_FILE = "ids.pt"
LAYER_NUM = 3

# Runtime knobs
RANDOM_STATE = 96
N_SPLITS_CV = 5
TEST_SIZE = 0.1


In [5]:
ids = load_out_tensor(output_dir, IDS_FILE)
print(ids.shape, ids.dtype)
ids = ids.cpu().numpy().astype(int)

torch.Size([41238]) torch.int64


Pick an i to test:

In [6]:
i = 96

Trying with Docked dataset:

In [ ]:
docked_ds = KinodataDocked()
docked_ds[0]

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCCC(C3CC(C4CCCC4)C4CCCCC34)C2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1',
  ligand={
    z=[28],
    x=[28, 12],
    pos=[28, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  (ligand, bond, ligand)={
    edge_index=[2, 64],
    edge_attr=[64, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  }
)

In [ ]:
print(f"{len(docked_ds)} , affinity at ident {docked_ds[i].ident.item()} : {docked_ds[i].y.item()}")
i = np.where(ids==docked_ds[i].ident.item())[0][0]

In [7]:
org_ds = KinodataDockedAgnostic()
org_ds_df = org_ds.data_frame
org_ds_df.head()

Loading raw data from /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/raw...
Reading data frame from /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/raw/kinodata_docked_v2.sdf.gz...
Deduping data frame (current size: 140977)...
138286 complexes remain after deduplication.
Checking for missing pocket mol2 files...


100%|██████████| 3551/3551 [00:08<00:00, 403.93it/s] 


Adding pocket sequences...
(138286, 25)


100%|██████████| 138286/138286 [00:00<00:00, 4196974.80it/s]


Exiting with 3552 cached sequences.
(138286, 26)
Converting to data list...
Done!


,docking.posit_probability,docking.chemgauss_score,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_units,compound_structures.canonical_smiles,...,UniprotID,similar.klifs_structure_id,similar.fp_similarity,ID,activities.standard_value,docking.predicted_rmsd,molecule,pocket_mol2_file,ident,structure.pocket_sequence
0,0.18,-13.526784,32335,CHEMBL817617,CHEMBL279,CHEMBL69638,nan,pIC50,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,...,P35968,5326,0.159664,LIG,5.148742,4.720892,<rdkit.Chem.rdchem.Mol object at 0x7f731533e340>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,0,KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVE...
1,0.24,-10.307055,32336,CHEMBL847682,CHEMBL4128,CHEMBL69638,nan,pIC50,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,...,Q02763,5553,0.2,32336,5.468521,5.696663,<rdkit.Chem.rdchem.Mol object at 0x7f731533dee0>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,1,DVIGEG__GQVLKAAIKRM____ELEVLCKLGPNIINLLGAYLAIE...
2,0.18,-11.764866,32680,CHEMBL677833,CHEMBL203,CHEMBL137635,nan,pIC50,nM,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,...,P00533,12838,0.212329,LIG,5.031517,4.851336,<rdkit.Chem.rdchem.Mol object at 0x7f731533e5e0>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,2,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLIMQ...
3,0.24,-10.21195,32770,CHEMBL674643,CHEMBL203,CHEMBL306988,nan,pIC50,nM,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)[O-])cc1,...,P00533,786,0.148936,LIG,3.301030,6.134686,<rdkit.Chem.rdchem.Mol object at 0x7f731533e6c0>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,3,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...
4,0.18,-3.132142,32773,CHEMBL675636,CHEMBL203,CHEMBL66879,nan,pKi,nM,O=C([O-])/C=C/c1ccc(O)cc1,...,P00533,15067,0.132075,LIG,3.000000,6.192890,<rdkit.Chem.rdchem.Mol object at 0x7f731533e730>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,4,KVLGS___GTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLIMQ...


In [24]:
id_idx = np.where(ids==i)[0][0]
print(id_idx)
ids[id_idx]

31914


96

In [25]:
ds_row = org_ds_df.loc[org_ds_df['ident']==i]
ds_row

,docking.posit_probability,docking.chemgauss_score,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_units,compound_structures.canonical_smiles,...,UniprotID,similar.klifs_structure_id,similar.fp_similarity,ID,activities.standard_value,docking.predicted_rmsd,molecule,pocket_mol2_file,ident,structure.pocket_sequence
96,0.81,-8.569431,66495,CHEMBL677833,CHEMBL203,CHEMBL137788,nan,pIC50,nM,C/N=N/Nc1ccc2ncnc(NCc3ccccc3)c2c1,...,P00533,786,0.297872,LIG,6.474955,1.248364,<rdkit.Chem.rdchem.Mol object at 0x7f5e1bc2e570>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,96,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...


In [18]:
skylines = org_ds_df[['ident', 'docking.chemgauss_score', 'activities.standard_value']].rename(columns={'docking.chemgauss_score': 'docking_score', 'activities.standard_value': 'affinity'})
skylines = skylines.set_index('ident')
print(len(skylines))
skylines.head()

138286


,docking_score,affinity
ident,,
0,-13.526784,5.148742
1,-10.307055,5.468521
2,-11.764866,5.031517
3,-10.21195,3.301030
4,-3.132142,3.000000


Loading another target to see how the ident and IDs are handled

In [8]:
test_target_name = "nitrogen_counts"
test_target = load_out_tensor(target_dir, f"{test_target_name}.pt")
print(type(test_target), len(test_target))
print(test_target[0])

<class 'dict'> 119522
4


In [ ]:
# Save skyline targets alongside other probing targets so they can be
# loaded uniformly via load_y_by_ids(target_dir=target_dir, targets_file=...).
for col in skylines.columns:
    save_out_tensor(skylines[col].to_dict(), output_dir=target_dir, filename=f"{col}.pt")
    print(f"Saved {col} -> {target_dir}/{col}.pt")

Test if it can be loaded:

In [ ]:
for col in skylines.columns:
    loaded_col = load_out_tensor(target_dir, f"{col}.pt")
    print(f"Loaded {col} from {target_dir}/{col}.pt")

Test if it can be loaded with ids:

In [6]:
from prob.paths_and_io import load_y_by_ids

loaded_y = load_y_by_ids(output_dir, target_dir=target_dir, targets_file="docking_score.pt")
print(type(loaded_y))
# print stats of y
pd.Series(loaded_y).describe()
# loaded_y[id_idx]

<class 'numpy.ndarray'>


count    4.123800e+04
mean     1.548838e+37
std      7.092711e+37
min     -2.873492e+01
25%     -1.364939e+01
50%     -1.112851e+01
75%     -8.504835e+00
max      3.402823e+38
dtype: float64

In [10]:
docking_score_dict = load_out_tensor(target_dir, "docking_score.pt")
print(len(docking_score_dict), "idents in docking_score.pt")

138286 idents in docking_score.pt


In [11]:
# docking.chemgauss_score encodes a failed/missing docking run as a huge sentinel
# (~3.4e38, float32 max) instead of NaN, and was never cast to float in
# process_raw_data() -- values in the dict are strings. Cast, then clean.
sentinel = np.float32(3.4028235e38)
docking_score_dict = {ident: float(v) for ident, v in docking_score_dict.items()}
n_sentinel = sum(1 for v in docking_score_dict.values() if abs(v) >= sentinel)
docking_score_dict = {
    ident: (np.nan if abs(v) >= sentinel else v)
    for ident, v in docking_score_dict.items()
}
print(f"cleaned {n_sentinel} sentinel values -> NaN")

cleaned 6523 sentinel values -> NaN


In [12]:
save_out_tensor(docking_score_dict, output_dir=target_dir, filename="docking_score.pt")
print(f"Saved docking_score -> {target_dir}/docking_score.pt")

Saved docking_score -> /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/targets/docking_score.pt


In [27]:
# i = 0
# expected = skylines.loc[int(ids[i].item()), "affinity"] # for docked ds
expected = skylines.loc[i, "affinity"] # for docked ds
assert loaded_y[id_idx] == expected, f"expected {expected} got {loaded_y[id_idx]}"

In [17]:
loaded_y = load_y_by_ids(output_dir, target_dir=target_dir, targets_file="affinity.pt")
print(type(loaded_y))
# print stats of y
pd.Series(loaded_y).describe()

assert len(loaded_y) == len(ids), f"len(loaded_y)={len(loaded_y)} != len(ids)={len(ids)}"
assert np.array(org_ds_df['activities.standard_value'][ids]).all() == np.array(loaded_y).all(), "loaded_y does not match the original dataset's activities.standard_value for the given ids"

<class 'numpy.ndarray'>


Now let's check X then!